# Skeleton-based Action Recognition (PoseC3D, GYM-limb)

Unlike `demo_recognition.ipynb` (which classifies raw RGB video directly), this model takes pose keypoints as
input. The pipeline is: extract frames -> detect people (`mmdet`) -> estimate pose keypoints (`mmpose`) -> classify
the skeleton sequence (`slowonly_r50_8xb16-u48-240e_gym-limb`, trained on the FineGym99 dataset).

In [ ]:
import tempfile
from pathlib import Path

import cv2
import mmcv
import torch
from mmengine import Config
from mmengine.utils import track_iter_progress

from mmaction.apis import (detection_inference, inference_skeleton,
                           init_recognizer, pose_inference)
from mmaction.registry import VISUALIZERS
from mmaction.utils import frame_extract

# This notebook lives in notebooks/, one level below the repository root.
ROOT = Path.cwd().parent
device = 'cuda:0'

In [ ]:
video = ROOT / "videos/backflip.mp4"

# Person detector (mmdet) — finds the bounding box(es) to run pose estimation on
det_config = ROOT / "configs/person_detector/faster-rcnn_r50_fpn_2x_coco_infer.py"
det_checkpoint = (
    "http://download.openmmlab.com/mmdetection/v2.0/faster_rcnn/"
    "faster_rcnn_r50_fpn_2x_coco/faster_rcnn_r50_fpn_2x_coco_"
    "bbox_mAP-0.384_20200504_210434-a5d8aa15.pth"
)

# Pose estimator (mmpose) — top-down COCO keypoints within each detected box
pose_config = ROOT / "configs/skeleton_coord/td-hm_hrnet-w32_8xb64-210e_coco-256x192_infer.py"
pose_checkpoint = (
    "https://download.openmmlab.com/mmpose/top_down/hrnet/"
    "hrnet_w32_coco_256x192-c78dce93_20200708.pth"
)

# Skeleton-based action recognizer (this repo)
skeleton_config = ROOT / "configs/skeleton/posec3d/slowonly_r50_8xb16-u48-240e_gym-limb.py"
skeleton_checkpoint = (
    "https://download.openmmlab.com/mmaction/v1.0/skeleton/posec3d/"
    "slowonly_r50_8xb16-u48-240e_gym-limb/"
    "slowonly_r50_8xb16-u48-240e_gym-limb_20220815-2e6e3c5c.pth"
)

label_map_path = ROOT / "tools/data/skeleton/label_map_gym99.txt"
labels = [line.strip() for line in label_map_path.read_text().splitlines()]

In [ ]:
# Extract frames into a temporary directory. Not a `with` block, since the
# extracted frame files need to stay on disk across the next few cells —
# cleaned up explicitly in the last cell of this notebook.
tmp_dir = tempfile.TemporaryDirectory()
frame_paths, frames = frame_extract(str(video), short_side=480, out_dir=tmp_dir.name)
h, w, _ = frames[0].shape
print(f'Extracted {len(frame_paths)} frames ({w}x{h})')

In [ ]:
det_results, _ = detection_inference(
    det_config, det_checkpoint, frame_paths,
    det_score_thr=0.9, det_cat_id=0, device=device,
)
torch.cuda.empty_cache()

In [ ]:
pose_results, pose_data_samples = pose_inference(
    pose_config, pose_checkpoint, frame_paths, det_results, device=device,
)
torch.cuda.empty_cache()

In [ ]:
model = init_recognizer(skeleton_config, skeleton_checkpoint, device=device)
pred_result = inference_skeleton(model, pose_results, (h, w))

top1_idx = int(pred_result.pred_score.argmax())
action_label = labels[top1_idx]
print(f'Predicted action: {action_label} ({pred_result.pred_score[top1_idx].item():.2f})')

In [ ]:
# Draw the detected skeleton on each frame and burn in the predicted action
# label, then display inline (no file left on disk).
pose_cfg = Config.fromfile(pose_config)
visualizer = VISUALIZERS.build(pose_cfg.visualizer)
visualizer.set_dataset_meta(pose_data_samples[0].dataset_meta)

FONTFACE = cv2.FONT_HERSHEY_DUPLEX
FONTSCALE = 0.75
FONTCOLOR = (255, 255, 255)  # BGR, white
THICKNESS = 1
LINETYPE = 1

vis_frames = []
for d, f in track_iter_progress(list(zip(pose_data_samples, frames))):
    f = mmcv.imconvert(f, 'bgr', 'rgb')
    visualizer.add_datasample(
        'result', f, data_sample=d,
        draw_gt=False, draw_heatmap=False, draw_bbox=True,
        show=False, wait_time=0, out_file=None, kpt_thr=0.3,
    )
    vis_frame = visualizer.get_image()
    cv2.putText(vis_frame, action_label, (10, 30), FONTFACE, FONTSCALE,
               FONTCOLOR, THICKNESS, LINETYPE)
    vis_frames.append(vis_frame)

In [ ]:
import moviepy.editor as mpy
from IPython.display import Video, display

with tempfile.TemporaryDirectory() as out_dir:
    out_path = Path(out_dir) / "demo_skeleton_out.mp4"
    vid = mpy.ImageSequenceClip(vis_frames, fps=24)
    vid.write_videofile(str(out_path), remove_temp=True, logger=None)
    # display() reads and embeds the file now, while it still exists;
    # the temp file is deleted once this block exits.
    display(Video(str(out_path), embed=True))

In [ ]:
# Clean up the extracted frames from the earlier frame_extract cell.
tmp_dir.cleanup()